In [0]:
from pyspark.sql.types import *
#from pyspark.sql.functions import date_format,to_timestamp,to_date, col, when, lit, year,month, dayofweek
from pyspark.sql import functions as F



In [0]:
bronze_table = spark.read.table('jc_demoworkspace.datahive.bronze_airline_departures')
display(bronze_table.head(10))

In [0]:
#changing times to timestamp to allow better sorting etc and adding a date to allow correct transformation
silver_df = bronze_table\
    .withColumn('scheduled_departure_time', F.to_timestamp(F.concat(F.col('date'),F.col('scheduled_departure_time')), "yyyy-MM-ddHH:mm"))\
    .withColumn('actual_departure_time', F.to_timestamp(F.concat(F.col('date'),F.col('actual_departure_time')), "yyyy-MM-ddHH:mm"))\
    .withColumn('wheelsoff_time', F.to_timestamp(F.concat(F.col('date'),F.col('wheelsoff_time')), "yyyy-MM-ddHH:mm"))
display(silver_df.limit(5))

In [0]:
#renaming schedule and actual elapsed so end users can decipher meaning easier

silver_df = silver_df\
    .withColumnRenamed('scheduled_elapsed_time_minutes', 'scheduled_flight_duration')\
    .withColumnRenamed('actual_elapsed_time_minutes', 'actual_flight_duration')
display(silver_df.head(10))


In [0]:
silver_df = silver_df\
    .withColumn('year', F.year(F.col('date')))\
    .withColumn('month', F.month(F.col('date')))\
    .withColumn('dayofweek', F.dayofweek(F.col('date')))\
    .withColumn('flight_num_date_key', F.concat(F.col('flight_number'),F.lit("_"),F.col('date')))\
    .withColumn('total_delay_minutes', F.col('delay_carrier_minutes') + F.col('delay_weather_minutes') + F.col('delay_national_aviation_system_minutes') + F.col('delay_security_minutes') + F.col('delay_late_aircraft_arrival_minutes'))\

#needed to have a usecase for when there are multiple equal primary delay reasons - may have been over kill as only 11 rows returned in Feb 2025
#going to move multiple delay and delay category to gold 
#Need to add a column concat of flightnumber & date as pk
display(silver_df.head(10))

In [0]:
silver_df_refined =\
 silver_df.filter(F.col('date').isNotNull())\
.drop('num_of_primary_delays')
display(silver_df_refined.head(10))

In [0]:
silver_df_refined.write.mode('overwrite').saveAsTable('jc_demoworkspace.datahive.silver_airline_departures')
spark.read.table('jc_demoworkspace.datahive.silver_airline_departures').limit(10)